In [105]:
import sys
sys.path.append('/data/bionets/og86asub/alternet-project/alternet/')
import pandas as pd
import alternet.data_preprocessing as preprocessing
from alternet.annotation import map_tf_ids
import alternet.annotation as annotation
import alternet.postprocessing as postprocessing

from alternet.inference import *
import yaml
import time
from typing import Any, Dict
import numpy as np

import seaborn as sns

def write_dict_to_yaml(data: Dict[str, Any], filepath: str):
    """Writes a Python dictionary to a YAML file."""

    with open(filepath, 'w') as f:
        yaml.dump(data, f, default_flow_style=False)




In [107]:
config_file = "/data/bionets/og86asub/alternet-project/alternet/configs/MAGNet_NF.yaml"

with open(config_file, 'r') as f:
    config = yaml.safe_load(f)


biomart_path = '/data/bionets/og86asub/alternet-project/alternet/data/biomart.txt'
tf_list_path = '/data/bionets/og86asub/alternet-project/alternet/data/allTFs_hg38.txt'
appris_df = pd.read_csv(config['appris'], sep='\t')

biomart = pd.read_csv(biomart_path, sep='\t')
tf_list = pd.read_csv(tf_list_path, sep='\t', header = None)
tf_list = map_tf_ids(tf_list, biomart)


gene_name_mapper = postprocessing.create_ensg_to_geneid_mapping(biomart)
transcript_name_mapper = postprocessing.create_enst_to_tid_mapping(biomart)

transcript_data = pd.read_csv(config['transcript_data'], index_col=0)

# Subset to protein coding isoforms
protein_coding_isoforms = list(appris_df[appris_df['Transcript type'] == 'protein_coding']['Transcript ID'])
transcript_data = transcript_data[transcript_data.transcript_id.isin(protein_coding_isoforms)]



gene_data = transcript_data.groupby('gene_id').sum()
gene_data = gene_data.drop(columns={'transcript_id'})

gene_data.index.name = 'gene_id'
gene_data = gene_data.reset_index()

sample_attributes = pd.read_csv(config['sample_attributes'])
sample_attributes = sample_attributes.loc[:, ['sample_name', 'etiology']]

conditions = ['DCM']

runs = 10
## Subset the samples of interest
for condi in conditions:

    samples = sample_attributes[sample_attributes['etiology'] == config['tissue']]
    samples = samples['sample_name'].tolist()

    gene_data_cp = gene_data.copy(deep=True)
    transcript_data_cp = transcript_data.copy(deep=True)
    ## Get unified gene and transcript table
    gene_data_cp = gene_data_cp.loc[:, ['gene_id'] + samples ]
    transcript_data_cp = transcript_data_cp.loc[:,['gene_id', 'transcript_id'] + samples]


    gene_target_names = list(tf_list['Gene stable ID'].unique())


    gene_data_cp = gene_data_cp.set_index('gene_id')
    gene_data_cp = gene_data_cp.T

    # scale data!!
    gene_data_cp_scaled = standardize_dataframe(gene_data_cp)

    transcript_data_cp = transcript_data_cp.set_index('transcript_id')
    transcript_data_cp = transcript_data_cp.drop('gene_id', axis=1)
    transcript_data_cp = transcript_data_cp.T

    # scale data!!
    transcript_data_cp_scaled = standardize_dataframe(transcript_data_cp)


    #start = time.monotonic()
    #canonical_grn = inference(gene_data=gene_data_cp, tf_list=gene_target_names, target_names = 'all', n_runs=runs)
    #end = time.monotonic()
    #runtime = {'canonical': end-start}
    
    #canonical_grn.to_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_canonical.tsv", header = True)

    #hybrid_data = create_hybrid_data(transcript_data_cp, gene_data_cp, tf_list)
    #hybrid_tf_names = list(tf_list['Transcript stable ID'].unique())
    #target_names = list(gene_data_cp.columns)

    #start = time.monotonic()
    #as_aware_grn = inference(gene_data=hybrid_data, tf_list=hybrid_tf_names, target_names=target_names, n_runs = runs)
    #runtime['as_aware'] = time.monotonic()-start

    #as_aware_grn.to_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_as_aware.tsv", header = True) 
    #write_dict_to_yaml(runtime, f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_runtime.yaml")

    # tf_list = pd.read_csv(tf_list_path, sep='\t', header = None)
    # tf_list = map_tf_ids(tf_list, biomart)

    # #filter aggregate
    # as_aware_grn = pd.read_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_as_aware.tsv", index_col=0)
    # canonical_grn = pd.read_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_canonical.tsv", index_col=0)

    # transcript_mapper = annotation.create_transcript_mapping(biomart)
    # as_aware_grn = postprocessing.map_transcript_to_gene(as_aware_grn, transcript_mapper)

    # as_aware_grn = postprocessing.create_edge_key(as_aware_grn)
    # canonical_grn = postprocessing.create_edge_key(canonical_grn, source_column='source', target_column='target')
    # canonical_grn = canonical_grn.rename(columns={'source': 'source_gene'})


    # # Categorize edges into gene-unique, isoform-unique, common
    # common_edges = postprocessing.get_common_edges(canonical_grn, as_aware_grn)
    # gene_unique, isoform_unique = postprocessing.get_diff(canonical_grn, as_aware_grn)

    # filtering_tracker = {'as_aware_base': as_aware_grn.shape[0], 'canonical_base': canonical_grn.shape[0],  'common_base': common_edges.shape[0], 'gene_unique_base': gene_unique.shape[0], 'isoform_unique_base': isoform_unique.shape[0]}

    # gene_unique, abs_threshold_g, f_gene = postprocessing.filter_aggregated(gene_unique, threshold_importance=0.2, threshold_frequency=10)
    # isoform_unique, abs_threshold_i, f_isoform = postprocessing.filter_aggregated(isoform_unique, threshold_importance=0.2, threshold_frequency=10)


    # filtering_tracker['gene_unique_filter_1'] = f_gene
    # filtering_tracker['isoform_unique_filter_1'] = f_isoform


    # isoform_categories = postprocessing.isoform_categorization(transcript_data_cp, gene_data_cp, tf_list)
    # isoform_categories_sub = isoform_categories.loc[:, ['Gene stable ID', 'Transcript stable ID', 'percentage', 'isoform_category']]

    # isoform_unique, iso_plausibility_filter  = postprocessing.plausibility_filtering(isoform_unique, isoform_categories_sub)
    # filtering_tracker['isoform_unique_plausibility_filter'] = iso_plausibility_filter
    
    # tf_database = annotation.create_transcipt_annotation_database(tf_list=tf_list, appris_path= config['appris'], digger_path=config['digger'])

    # isoform_unique = annotation.merge_annotations_to_grn(isoform_unique, tf_database, transcript_column='source_transcript')
    # isoform_unique  = isoform_unique.drop(columns={'Gene stable ID', 'Transcript stable ID', 'Transcript type'})
    # isoform_unique = isoform_unique.rename(columns = {'target': 'target_gene'})
    # isoform_unique = isoform_unique.add_gene_names(isoform_unique, biomart)
    # isoform_unique.to_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_unique_isofoms.tsv")

    # gene_categories = postprocessing.get_gene_cases(isoform_categories_sub)
    # gene_unique, filtering_info_gene_plaus = postprocessing.plausibility_filtering_gene_unique(gene_unique, gene_categories)
    # filtering_tracker['gene_unique_plausibility_filter'] = filtering_info_gene_plaus
    # gene_unique = gene_unique.rename(columns = {'target': 'target_gene'})
    # gene_unique = postprocessing.add_gene_names(gene_unique, biomart)
    # gene_unique.to_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_unique_genes.tsv")


    # merged_edges = postprocessing.create_common_edge_dataframe(common_edges)
    # consistent, ambigous = postprocessing.split_by_isoform_category(merged_edges, gene_categories)
    

    # consistent, filter_info = postprocessing.plausibility_filtering_common_edges_dominant(consistent)
    # filtering_tracker['common_dominant_filter'] = filter_info
    # consistent = postprocessing.add_gene_names(consistent, biomart)
    # consistent.to_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_consistent_both.tsv")


    # likely_isoform_specific, ambigous, filter_info_iu = postprocessing.find_likely_isoform_specific(ambigous)
    # filtering_tracker['common_isoform_likely_filter'] = filter_info_iu
    # likely_isoform_specific = annotation.merge_annotations_to_grn(likely_isoform_specific, tf_database, transcript_column='source_transcript')
    
    # likely_isoform_specific = postprocessing.add_gene_names(likely_isoform_specific, biomart)
    # likely_isoform_specific.to_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_likely_isoform_specific.tsv")

    
    # likely_gene_specific, ambigous, filter_infor_gu = postprocessing.find_likely_gene_specific(ambigous)
    # filtering_tracker['common_gene_likely_filter'] = filter_infor_gu
    
    # likely_gene_specific = postprocessing.add_gene_names(edgelist = likely_gene_specific, biomart=biomart)
    # likely_gene_specific.to_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_likely_gene_specific.tsv")

    # ambigous = postprocessing.add_gene_names(edgelist=ambigous, biomart=biomart)
    # ambigous.to_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_ambigous.tsv")

    # gene_to_transcript_mapping = annotation.create_filtered_gene_to_transcripts_mapping(biomart, gene_list = gene_data_cp_scaled.columns, transcript_list = transcript_data_cp_scaled.columns)
    # correlation_collector = annotation.compute_isoform_gene_correlations(transcript_data_cp_scaled, gene_data_cp_scaled, gene_to_transcript_mapping)

        
    # write_dict_to_yaml(filtering_tracker, f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/{condi}_filtering_tracker.yaml")






In [1]:
import sys
sys.path.append('/data/bionets/og86asub/alternet-project/alternet/')
import pandas as pd
import alternet.data_preprocessing as preprocessing
from alternet.annotation import map_tf_ids
import alternet.annotation as annotation
import alternet.postprocessing as postprocessing

from alternet.inference import *
import yaml
import time
from typing import Any, Dict
import numpy as np

import seaborn as sns

def write_dict_to_yaml(data: Dict[str, Any], filepath: str):
    """Writes a Python dictionary to a YAML file."""

    with open(filepath, 'w') as f:
        yaml.dump(data, f, default_flow_style=False)




/data/bionets/og86asub/alternet-project/alternet/.pixi/envs/kernel/lib/python3.11/site-packages/numba/core/decorators.py:248: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)


In [2]:
config_file = "/data/bionets/og86asub/alternet-project/alternet/configs/MAGNet_NF.yaml"

with open(config_file, 'r') as f:
    config = yaml.safe_load(f)


biomart_path = '/data/bionets/og86asub/alternet-project/alternet/data/biomart.txt'
tf_list_path = '/data/bionets/og86asub/alternet-project/alternet/data/allTFs_hg38.txt'
appris_df = pd.read_csv(config['appris'], sep='\t')

biomart = pd.read_csv(biomart_path, sep='\t')
tf_list = pd.read_csv(tf_list_path, sep='\t', header = None)
tf_list = map_tf_ids(tf_list, biomart)

tf_list = pd.read_csv(tf_list_path, sep='\t', header = None)
tf_list = map_tf_ids(tf_list, biomart)
tf_database = annotation.create_transcipt_annotation_database(tf_list=tf_list, appris_path= config['appris'], digger_path=config['digger'])


/data/bionets/og86asub/alternet-project/alternet/src/alternet/annotation.py:91: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  digger = pd.read_csv(digger_path)


In [73]:
appris_df = pd.read_csv(config['appris'], sep='\t')



In [74]:
appris_df

,Ensembl Gene ID,Gene name (HGNC),Transcript ID,Translation ID,Is translated?,Transcript type,Not found tag,CCDS ID,Transcript support level,Protein length,...,Structure score (Matador),Conservation (CORSAIR),Domain Score (SPADE),Trans-membrane helices (THUMP),Signal sequence (CRASH),Trifid Score,Peptides,APPRIS score,APPRIS Annotation,MANE
0,ENSG00000186510,CLCNKA,ENST00000491433,-,NO_TRANSLATION,protein_coding_CDS_not_defined,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ENSG00000186510,CLCNKA,ENST00000331433,ENSP00000332771,TRANSLATION,protein_coding,-,CCDS167.1,1,687.0,...,15.05,7.2,273.6,12.0,"-4,-6",0.9990,-,10.000,ALTERNATIVE:1,MANE_Select
2,ENSG00000186510,CLCNKA,ENST00000439316,ENSP00000414445,TRANSLATION,protein_coding,-,CCDS57973.1,2,644.0,...,14.05,3.2,264.4,12.0,"-4,-5",0.3248,-,6.712,MINOR,NaN
3,ENSG00000186510,CLCNKA,ENST00000464764,-,NO_TRANSLATION,protein_coding_CDS_not_defined,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ENSG00000186510,CLCNKA,ENST00000375692,ENSP00000364844,TRANSLATION,protein_coding,-,CCDS41269.1,1,686.0,...,14.90,7.2,273.6,12.0,"-4,-6",1.0000,-,9.700,PRINCIPAL:4,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170707,ENSG00000198695,MT-ND6,ENST00000361681,ENSP00000354665,TRANSLATION,protein_coding,stop,-,-,174.0,...,1.00,4.2,59.1,0.0,"0,-5",0.7910,1,-2.000,PRINCIPAL:1,NaN
170708,ENSG00000228253,MT-ATP8,ENST00000361851,ENSP00000355265,TRANSLATION,protein_coding,-,-,-,68.0,...,0.60,1.0,68.6,0.0,"-2,-5",0.5435,5,12.000,PRINCIPAL:1,NaN
170709,ENSG00000198888,MT-ND1,ENST00000361390,ENSP00000354687,TRANSLATION,protein_coding,start/stop,-,-,318.0,...,1.00,2.5,352.9,0.0,"1,-3",0.7302,5,-2.000,PRINCIPAL:1,NaN
170710,ENSG00000198886,MT-ND4,ENST00000361381,ENSP00000354961,TRANSLATION,protein_coding,stop,-,-,459.0,...,1.00,10.6,265.4,0.0,"0,-4",1.0000,7,-2.000,PRINCIPAL:1,NaN


In [ ]:

def get_common_annotation_dataframe(unique_genes, annotation_database):
    '''
    Build a transcript-level annotation table for a set of unique transcription factor (TF) isoforms.

    Parameters:

        unique_tfs : list
            List of Ensembl transcript IDs corresponding to unique TF isoforms.

        annotation_database : pd.DataFrame
            DataFrame containing APPRIS, DIGGER, and Ensembl annotation data.

    Returns:
    
        pd.DataFrame
            DataFrame indexed by transcript ID, containing annotation information for each TF isoform.

    
    '''
    # Precompute annotations for all transcripts
    annotations = []
    for gid in unique_genes.keys():
        annot = get_common_annotations(gid, unique_genes[gid], annotation_database)
        annot['Gene stable ID'] = gid
        annotations.append(annot)
    annotation_df = pd.DataFrame(annotations).set_index('Gene stable ID')
    return annotation_df



def annotate_gene_exclusive_edges(grn, annotation_database, gene_transcript_mapping, gene_column = 'source_gene', transcript_column = 'source_transcript'):
    '''
    Merge transcript-level annotations into a gene regulatory network (GRN) based on source transcript IDs.

    Parameters:

        grn : pd.DataFrame
            DataFrame representing the gene regulatory network, containing a 'source' column with transcript IDs.

        annotation_database : pd.DataFrame
            DataFrame containing APPRIS, and DIGGER annotation data.

    Returns:
    
        pd.DataFrame
            GRN DataFrame with additional columns from the annotation database merged on the 'source' transcript ID.

    '''

    #gene_transcript_mapping = grn.groupby(gene_column)[transcript_column].unique().apply(list).to_dict()
    annot_df = get_common_annotation_dataframe(gene_transcript_mapping, annotation_database)
    grn_annot = grn.merge(annot_df, how='left', left_on=gene_column, right_index=True)
    return grn_annot



def get_intersection(relevant_items):

    relevant_items = [set(tl)  for tl in relevant_items]

    if len(relevant_items)>1:
        common_elements = relevant_items[0].intersection(*relevant_items[1:])
    elif len(relevant_items)>0:
        common_elements = relevant_items[0]
    else:
        common_elements = []
    return list(common_elements)



def get_intersection_wrapper(transcript_data, transcript_info):
    '''
    Compares 'Exon stable ID' and 'Pfam ID' from a transcript dictionary and a DataFrame
    of related isoforms, and returns a dictionary with unique values for each of these columns.

    Parameters:

        transcript_dict : dict
            Dictionary containing 'Exon stable ID' and 'Pfam ID' for a single transcript.

        isoforms_df : pd.DataFrame
            DataFrame containing related isoforms with 'Exon stable ID' and 'Pfam ID' columns.

    Returns:
    
        dict
            A dictionary with keys 'unique_exons' and 'unique_pfam', containing lists of unique values.
    '''


    for col, unique_key in [('Exon stable ID', 'common Exon stable ID'),
                            ('Pfam ID', 'common Pfam ID')]:
        transcript_data[unique_key] = get_intersection(transcript_info.get(col).dropna())

    return transcript_data


def get_common_annotations(gene_id, transcript_ids, annotation_database):

    '''
    Get the available annotations of the transcript and compile a unique list of exon IDs and Pfam IDs.

    Parameters:

        transcript_id : str
            Ensembl ID of the transcript.

        annotation_database : pd.DataFrame
            DataFrame containing APPRIS, and DIGGER annotation data.

    Returns:
    
        dict
            A dictionary containing all annotations of the transcript, including unique exon IDs and Pfam IDs.
  
    '''

    gene = annotation_database[annotation_database['Gene stable ID'] == gene_id]

    if gene.empty:
        # return same structure with None values if transcript not found
        return pd.Series({
            'Protein Coding': False,
            'Transcript type': None,
            'APPRIS Annotation': None,
            'Exon stable ID': None,
            'common Exon stable ID': None,
            'Pfam ID': None,
            'common Pfam ID': None
        })
    

    
    relevant_transcripts = gene[gene['Transcript stable ID'].isin(transcript_ids)]

    if relevant_transcripts.shape[0]==0:
        return pd.Series({
            'Protein Coding': False,
            'Transcript type': None,
            'APPRIS Annotation': None,
            'Exon stable ID': None,
            'common Exon stable ID': None,
            'Pfam ID': None,
            'common Pfam ID': None
        })

    g_row = relevant_transcripts.iloc[0]

    transcript_data = {
        'Protein Coding': True,
        'Transcript type': g_row.get('Transcript type'),
        'APPRIS Annotation': g_row.get('APPRIS Annotation'),
        'Trifid Score': g_row.get('Trifid Score'),
        'Exon stable ID': g_row.get('Exon stable ID'),
        'Pfam ID': g_row.get('Pfam ID'),
    }


    # for compare the transcript data against the chosen set to find out what makes the transcript unique.
    unique_items = get_intersection_wrapper(transcript_data, relevant_transcripts)
    return pd.Series(unique_items)



In [113]:
condi = 'NF'
ambigous = pd.read_csv(f"/data/bionets/og86asub/alternet-project/alternet-manuscript/results/DCM_unique_genes.tsv")


In [115]:


gene_to_transcript_mapping = annotation.create_filtered_gene_to_transcripts_mapping(biomart, gene_list = gene_data_cp_scaled.columns, transcript_list = transcript_data_cp_scaled.columns)

ambigous = annotate_gene_exclusive_edges(ambigous, annotation_database=tf_database, gene_transcript_mapping=gene_to_transcript_mapping)

In [116]:
ambigous

,Unnamed: 0,source_gene,target_gene,frequency,mean_importance,median_importance,edge_key,target_gene_name,Protein Coding,Transcript type,APPRIS Annotation,Exon stable ID,unique Exon stable ID,Pfam ID,unique Pfam ID,Trifid Score,common Exon stable ID,common Pfam ID
0,13939188,ENSG00000185591,ENSG00000118194,10,5.738032,6.113752,ENSG00000185591_ENSG00000118194,TNNT2,True,protein_coding,PRINCIPAL:4,"[ENSE00002328883, ENSE00003476492, ENSE0000132...",NaN,"[PF00096, PF00096, PF00096]",NaN,1.0000,"[ENSE00001262751, ENSE00001300823, ENSE0000126...",[PF00096]
1,13943390,ENSG00000185591,ENSG00000162623,10,6.612220,5.459664,ENSG00000185591_ENSG00000162623,TYW3,True,protein_coding,PRINCIPAL:4,"[ENSE00002328883, ENSE00003476492, ENSE0000132...",NaN,"[PF00096, PF00096, PF00096]",NaN,1.0000,"[ENSE00001262751, ENSE00001300823, ENSE0000126...",[PF00096]
2,7195009,ENSG00000140262,ENSG00000100292,10,4.606507,4.218943,ENSG00000140262_ENSG00000100292,HMOX1,True,protein_coding,ALTERNATIVE:1,"[ENSE00003568449, ENSE00003606443, ENSE0000365...",NaN,[PF00010],NaN,0.9741,"[ENSE00003523777, ENSE00003489614, ENSE0000363...",[PF00010]
3,1972368,ENSG00000092820,ENSG00000198417,10,3.346036,3.361327,ENSG00000092820_ENSG00000198417,MT1F,True,protein_coding,PRINCIPAL:1,"[ENSE00001212701, ENSE00003532760, ENSE0000353...",NaN,"[PF09379, PF09379, PF00373, PF09379, PF00373, ...",NaN,1.0000,"[ENSE00003527117, ENSE00001028545, ENSE0000102...","[PF09379, PF09380, PF00769, PF00373]"
4,10163778,ENSG00000166716,ENSG00000126243,10,3.298565,2.976567,ENSG00000166716_ENSG00000126243,LRFN3,True,protein_coding,PRINCIPAL:1,"[ENSE00002553930, ENSE00002548999, ENSE0000255...",NaN,"[PF00096, PF00096, PF00096, PF16622]",NaN,1.0000,"[ENSE00003533064, ENSE00003654939, ENSE0000355...","[PF16622, PF00096]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69,8113671,ENSG00000148308,ENSG00000152422,10,0.616231,0.581777,ENSG00000148308_ENSG00000152422,XRCC4,True,protein_coding,MINOR,"[ENSE00001372100, ENSE00003789769, ENSE0000164...",NaN,"[PF09734, PF09734, PF09734]",NaN,0.0370,"[ENSE00001372100, ENSE00003789769, ENSE0000351...",[PF09734]
70,17252007,ENSG00000256087,ENSG00000134007,10,1.634007,0.566697,ENSG00000256087_ENSG00000134007,ADAM20,True,protein_coding,PRINCIPAL:1,"[ENSE00001124026, ENSE00002515591, ENSE0000251...",NaN,"[PF01352, PF01352, PF00096, PF00096, PF00096, ...",NaN,1.0000,"[ENSE00002463048, ENSE00001124026, ENSE0000251...","[PF00096, PF01352]"
71,14742325,ENSG00000196109,ENSG00000254709,10,0.771264,0.550638,ENSG00000196109_ENSG00000254709,IGLL5,True,protein_coding,MINOR,"[ENSE00002501651, ENSE00002496192, ENSE0000251...",NaN,"[PF00096, PF00096, PF00096, PF00096, PF00096, ...",NaN,0.4670,[ENSE00002496192],"[PF00096, PF13465]"
72,10718651,ENSG00000168813,ENSG00000119121,10,0.580384,0.548661,ENSG00000168813_ENSG00000119121,TRPM6,True,protein_coding,PRINCIPAL:1,"[ENSE00001355115, ENSE00003695352, ENSE0000132...",NaN,"[PF00096, PF00096]",NaN,1.0000,"[ENSE00001185554, ENSE00003695352, ENSE0000132...",[PF00096]


In [101]:
ambigous['common Pfam ID'].value_counts()

common Pfam ID
[PF00096]                      86068
[PF01352]                      80706
[]                             77526
[PF00096, PF01352]             50312
[PF00010]                      20831
                               ...  
[PF13606, PF12796]               951
[PF05225]                        894
[PF00452]                        839
[PF00096, PF01352, PF13894]      669
[PF05063]                        647
Name: count, Length: 190, dtype: int64

In [25]:
ambigous.groupby('source_gene')['source_transcript'].unique().apply(list).to_dict()

{'ENSG00000001497': ['ENST00000374807', 'ENST00000374811', 'ENST00000469091'],
 'ENSG00000005889': ['ENST00000379188', 'ENST00000539115'],
 'ENSG00000006468': ['ENST00000405192', 'ENST00000405358', 'ENST00000430479'],
 'ENSG00000006704': ['ENST00000424337', 'ENST00000470715', 'ENST00000476977'],
 'ENSG00000007866': ['ENST00000338863', 'ENST00000639578'],
 'ENSG00000010244': ['ENST00000321233', 'ENST00000394670', 'ENST00000394673'],
 'ENSG00000012048': ['ENST00000352993', 'ENST00000357654', 'ENST00000634433'],
 'ENSG00000013441': ['ENST00000321356', 'ENST00000409769'],
 'ENSG00000018189': ['ENST00000226328', 'ENST00000381006', 'ENST00000417478'],
 'ENSG00000018869': ['ENST00000301310', 'ENST00000589143'],
 'ENSG00000019485': ['ENST00000534751', 'ENST00000622142'],
 'ENSG00000020256': ['ENST00000371518', 'ENST00000371523'],
 'ENSG00000025156': ['ENST00000368455', 'ENST00000452194'],
 'ENSG00000025434': ['ENST00000441012', 'ENST00000467728', 'ENST00000481889'],
 'ENSG00000028839': ['ENST0